# 4. Full-corpus inference, sentiment audit and issue profiles

| | |
|---|---|
| **Stage** | 4 of 9 |
| **Purpose** | Apply the Task 1 and Task 2 models to every review, score every detected (sentence, aspect) pair with Task 3, sample pairs for the human sentiment audit, retrain Task 3 on gold plus audit labels (version 2), and compute the first issue profiles and the first dashboard files. |
| **Inputs** | `models/task1_bert_combined, models/task2_final, models/task3_final`<br>`outputs/deploy_config.json`<br>`data/processed/{game}_clean.csv`<br>`data/annotation_v2/gold_set_combined.csv`<br>`outputs/sentiment_audit.csv (after the manual audit)` |
| **Outputs** | `outputs/predictions/{game}_aspects.parquet`<br>`outputs/predictions/{game}_pairs.parquet`<br>`outputs/dashboard/overall_{game}.parquet (and the first dashboard tables)`<br>`outputs/sentiment_audit.csv`<br>`models/task3_final_v2, outputs/task3_oof_*.npy`<br>`outputs/issue_profile_all.csv, outputs/issue_profile_robustness.csv` |
| **Runtime** | GPU required (Colab T4). |
| **Depends on** | Stage 3 |
| **Configuration** | Set `PROJECT_ROOT` in the first code cell if your Drive folder is not `MyDrive/GameSense`. |

> **Note.** The sentence-level aspect predictions and the review-level overall sentiment written here are used unchanged by notebooks 07 and 09. The Task 3 version 2 model is superseded by notebook 07.

Environment, configuration, and file checks

In [ ]:
%pip install -q -U transformers accelerate

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, gc, time
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = '/content/drive/MyDrive/GameSense'
assert torch.cuda.is_available(), "GPU：Runtime -> Change runtime type -> T4 GPU，rerun"
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

os.makedirs(f'{PROJECT_ROOT}/outputs/predictions', exist_ok=True)

# Deployment configuration: Create using default values ​​if it does not exist.
cfg_path = f'{PROJECT_ROOT}/outputs/deploy_config.json'
if os.path.exists(cfg_path):
    with open(cfg_path) as f:
        CFG = json.load(f)
else:
    CFG = {"task2_threshold": 0.30,
           "task2_model_dir": "models/task2_final",
           "task3_model_dir": "models/task3_final",
           "apply_applicability_mask": True}
    with open(cfg_path, 'w') as f:
        json.dump(CFG, f, indent=2)
    print("deploy_config.json does not exist; created using default values.")

TASK2_DIR = f'{PROJECT_ROOT}/{CFG["task2_model_dir"]}'
TASK3_DIR = f'{PROJECT_ROOT}/{CFG["task3_model_dir"]}'
THRESHOLD = CFG["task2_threshold"]
print("Deployment Configuration:", CFG)

sys.path.insert(0, f'{PROJECT_ROOT}/scripts')
from aspect_taxonomy import ASPECT_TAXONOMY, ASPECT_IDS

GAME_SLUGS = ["royalmatch", "genshin", "mlbb", "pubgm", "arknights"]
checks = {"Task2 final model": f'{TASK2_DIR}/model.safetensors',
          "Task3 final model": f'{TASK3_DIR}/model.safetensors'}
for slug in GAME_SLUGS:
    checks[f"{slug} Sentence data"] = f'{PROJECT_ROOT}/data/processed/{slug}_sentences.parquet'
for name, path in checks.items():
    print("✅" if os.path.exists(path) else "❌", name)

Applicability mapping, reading sentences

In [ ]:
COMMON = ["performance_stability", "monetization_value", "advertisement",
          "level_difficulty_balance", "content_progression", "paywall", "account_support"]
GENRE = {"royalmatch": [], "genshin": ["gacha_rates", "character_operator_balance"],
         "mlbb": ["character_operator_balance", "matchmaking", "cheating"],
         "pubgm": ["cheating", "matchmaking"],
         "arknights": ["gacha_rates", "character_operator_balance", "grind_progression"]}

def applicable_vector(slug):
    allowed = set(COMMON) | set(GENRE[slug])
    return np.array([a in allowed for a in ASPECT_IDS])

def load_sentences(slug):
    for name in [f'{slug}_sentences.parquet', f'{slug}_sentences_weak.parquet']:
        p = f'{PROJECT_ROOT}/data/processed/{name}'
        if os.path.exists(p):
            df = pd.read_parquet(p)
            break
    else:
        raise FileNotFoundError(slug)
    df = df.copy()
    df["game"] = slug
    df["sentence_uid"] = df["reviewId"].astype(str) + "_" + df["sentence_idx"].astype(str)
    df["sentence_text"] = df["sentence_text"].fillna("").astype(str)
    keep = [c for c in ["sentence_uid", "reviewId", "game", "sentence_text", "review_score",
                        "review_overall_sentiment_weak", "reviewCreatedVersion", "at"]
            if c in df.columns]
    return df[keep].reset_index(drop=True)

sent = {slug: load_sentences(slug) for slug in GAME_SLUGS}
for slug, d in sent.items():
    print(f"{slug:12s} {len(d):6d} sentences")
print("Total:", sum(len(d) for d in sent.values()))

Load the Task 2 model and the batch inference function

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tok2 = AutoTokenizer.from_pretrained(TASK2_DIR)
m2 = AutoModelForSequenceClassification.from_pretrained(TASK2_DIR).to(device).eval()

@torch.inference_mode()
def predict_logits(texts, tok, model, batch_size=128, max_length=96, pair_texts=None):
    order = np.argsort([len(t) for t in texts])      # Sort by length to reduce padding and improve speed.
    out = None
    for start in range(0, len(texts), batch_size):
        idx = order[start:start + batch_size]
        batch = [texts[i] for i in idx]
        if pair_texts is None:
            enc = tok(batch, truncation=True, max_length=max_length,
                      padding=True, return_tensors="pt")
        else:
            enc = tok(batch, [pair_texts[i] for i in idx], truncation=True,
                      max_length=max_length, padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**enc).logits
        logits = logits.float().cpu().numpy()
        if out is None:
            out = np.zeros((len(texts), logits.shape[1]), dtype=np.float32)
        out[idx] = logits
    return out

# The tag order must match the taxonomy; otherwise, the results for each aspect will be misaligned
labels_in_model = [m2.config.id2label[i] for i in range(m2.config.num_labels)]
assert labels_in_model == list(ASPECT_IDS), "The model label order is inconsistent with ASPECT_IDS; please pause and check."
print("Task2 Model Loaded ✅   target number:", m2.config.num_labels, " consistent tag order ✅")

Run inference on five games and save the results

In [ ]:
def run_task2(slug):
    t0 = time.time()
    df = sent[slug]
    logits = predict_logits(df["sentence_text"].tolist(), tok2, m2)
    probs = 1 / (1 + np.exp(-logits))

    app = applicable_vector(slug)
    masked = probs * app[None, :] if CFG.get("apply_applicability_mask", True) else probs
    pred = masked >= THRESHOLD

    prob_df = pd.DataFrame(probs.astype(np.float32), columns=[f"p_{a}" for a in ASPECT_IDS])
    pred_df = pd.DataFrame(pred, columns=[f"pred_{a}" for a in ASPECT_IDS])
    out = pd.concat([df.reset_index(drop=True), prob_df, pred_df], axis=1)
    out.to_parquet(f'{PROJECT_ROOT}/outputs/predictions/{slug}_aspects.parquet', index=False)

    rate = pred.sum() / (len(df) * app.sum())
    any_rate = pred.any(axis=1).mean()
    print(f"{slug:12s} {len(df):6d} sentences | {(time.time() - t0) / 60:.1f} mins | "
          f"True positive rate within the applicable category={rate:.3f} | Proportion of sentences containing at least one aspect={any_rate:.3f}")
    return out

preds_all = {slug: run_task2(slug) for slug in GAME_SLUGS}

print("\nThe aspect with the most hits in each game（Top 5）:")
for slug, out in preds_all.items():
    counts = pd.Series({a: int(out[f"pred_{a}"].sum()) for a in ASPECT_IDS}).sort_values(ascending=False)
    print(f"  {slug:12s}", dict(counts.head(5)))

Check 1: Spot-check for precision.

In [ ]:
def sample_predicted(slug, aspect, n=20, seed=1):
    out = preds_all[slug]
    d = out[out[f"pred_{aspect}"]]
    d = d.sample(n=min(n, len(d)), random_state=seed)
    print(f"===== {slug} / {aspect}  (Hit {int(out[f'pred_{aspect}'].sum())} sentences，extract {len(d)} sentences) =====")
    for _, r in d.iterrows():
        print(f"p={r[f'p_{aspect}']:.2f} | {r['sentence_text'][:140]}")
    print()

sample_predicted("genshin", "content_progression")
sample_predicted("arknights", "content_progression")
sample_predicted("mlbb", "matchmaking", n=10)      # Control group: High expected precision

Check 2: Are there any missed detections of cheating in PUBG

In [ ]:
kw = ["cheat", "hack", "aimbot", "wallhack", "modded"]
pub = preds_all["pubgm"]
m = pub["sentence_text"].str.lower().apply(lambda s: any(k in s for k in kw))
print("PUBG Sentences containing cheating-related keywords:", int(m.sum()))
print("Among them, the model classified as cheating:", int((m & pub["pred_cheating"]).sum()))
print("Total number of instances where the model was judged to be cheating:", int(pub["pred_cheating"].sum()))

miss = pub[m & ~pub["pred_cheating"]]
print("\nExamples containing keywords that the model did not classify as cheating.:")
for _, r in miss.sample(n=min(10, len(miss)), random_state=1).iterrows():
    print(f"p={r['p_cheating']:.2f} | {r['sentence_text'][:140]}")

Emotional assessment

Load Task 3 and run inference

In [ ]:
tok3 = AutoTokenizer.from_pretrained(TASK3_DIR)
m3 = AutoModelForSequenceClassification.from_pretrained(TASK3_DIR).to(device).eval()
SENT_LABELS = [m3.config.id2label[i] for i in range(m3.config.num_labels)]
assert SENT_LABELS == ["negative", "neutral", "positive"], SENT_LABELS
print("Task3 model loaded ✅", SENT_LABELS)

def run_task3(slug):
    t0 = time.time()
    out = preds_all[slug]
    parts = []
    for a in ASPECT_IDS:
        idx = np.where(out[f"pred_{a}"].values)[0]
        if len(idx):
            parts.append(pd.DataFrame({"row": idx, "aspect": a}))
    pairs = pd.concat(parts, ignore_index=True)

    texts = out["sentence_text"].values[pairs["row"].values].tolist()
    aspect_text = [ASPECT_TAXONOMY[a]["label"] for a in pairs["aspect"]]
    logits = predict_logits(texts, tok3, m3, pair_texts=aspect_text)
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = e / e.sum(axis=1, keepdims=True)

    meta_cols = [c for c in ["sentence_uid", "reviewId", "game", "sentence_text", "review_score",
                             "reviewCreatedVersion", "at"] if c in out.columns]
    res = out[meta_cols].iloc[pairs["row"].values].reset_index(drop=True)
    res["aspect"] = pairs["aspect"].values
    res["p_aspect"] = [out[f"p_{a}"].values[r] for a, r in zip(pairs["aspect"], pairs["row"])]
    res["sentiment"] = [SENT_LABELS[i] for i in probs.argmax(axis=1)]
    for j, lab in enumerate(SENT_LABELS):
        res[f"p_{lab}"] = probs[:, j].astype(np.float32)

    res.to_parquet(f'{PROJECT_ROOT}/outputs/predictions/{slug}_pairs.parquet', index=False)
    print(f"{slug:12s} {len(res):6d} (sentences,aspect)pairs | {(time.time() - t0) / 60:.1f} mins")
    return res

pairs_all = {slug: run_task3(slug) for slug in GAME_SLUGS}

allp = pd.concat(pairs_all.values(), ignore_index=True)
print("\n=== Emotional distribution across games ===")
print(pd.crosstab(allp["game"], allp["sentiment"], normalize="index").round(3).to_string())
print("\n=== Proportion of negative feedback by aspect (all games combined) ===")
tab = allp.groupby("aspect")["sentiment"].agg(
    n="size", neg=lambda s: (s == "negative").mean()).sort_values("n", ascending=False)
print(tab.round(3).to_string())

#### Audit code

In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/GameSense'
sys.path.insert(0, f'{PROJECT_ROOT}/scripts')
from aspect_taxonomy import ASPECT_TAXONOMY, ASPECT_IDS

GAME_SLUGS = ["royalmatch", "genshin", "mlbb", "pubgm", "arknights"]
AUDIT_PATH = f'{PROJECT_ROOT}/outputs/sentiment_audit.csv'

def load_pairs():
    parts = []
    for slug in GAME_SLUGS:
        p = f'{PROJECT_ROOT}/outputs/predictions/{slug}_pairs.parquet'
        if not os.path.exists(p):
            raise FileNotFoundError(f"can not find {p}，please run notebook again ")
        parts.append(pd.read_parquet(p))
    d = pd.concat(parts, ignore_index=True)
    d["stars"] = pd.to_numeric(d["review_score"], errors="coerce")
    return d

pairs = load_pairs()
print("(sentences, aspect) total pairs:", len(pairs))
print(pairs.groupby("aspect").size().sort_values(ascending=False).to_string())

Generate audit files

In [ ]:
def generate_audit(force=False):
    if os.path.exists(AUDIT_PATH):
        old = pd.read_csv(AUDIT_PATH)
        n_done = int(old["human_label"].fillna("").astype(str).ne("").sum())
        if n_done > 0 and not force:
            print(f"Already exsits {n_done} records，do not cover。")
            return old

    gold_ids = set(pd.read_csv(f'{PROJECT_ROOT}/data/annotation_v2/gold_set_combined.csv')["sentence_uid"])
    pool = pairs[~pairs["sentence_uid"].isin(gold_ids)]        # Exclude sentences already in the gold set.

    def sample_cell(d, q, seed=42):
        lo, hi = d[d["stars"] <= 2], d[d["stars"] >= 4]
        a = lo.sample(n=min(q // 2, len(lo)), random_state=seed)
        b = hi.sample(n=min(q - len(a), len(hi)), random_state=seed)
        got = pd.concat([a, b])
        if len(got) < q:
            rest = d.drop(got.index)
            got = pd.concat([got, rest.sample(n=min(q - len(got), len(rest)), random_state=seed)])
        return got

    quota = {("genshin", "gacha_rates"): 40, ("arknights", "gacha_rates"): 20,
             ("genshin", "content_progression"): 30, ("arknights", "content_progression"): 20,
             ("royalmatch", "content_progression"): 7, ("mlbb", "content_progression"): 7,
             ("pubgm", "content_progression"): 6,
             ("genshin", "character_operator_balance"): 12,
             ("mlbb", "character_operator_balance"): 10,
             ("arknights", "character_operator_balance"): 8}
    for g_ in GAME_SLUGS:
        quota[(g_, "monetization_value")] = 6
        quota[(g_, "level_difficulty_balance")] = 6

    parts = [sample_cell(pool[(pool["game"] == g_) & (pool["aspect"] == a)], q)
             for (g_, a), q in quota.items()]
    audit = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)
    audit = audit[["sentence_uid", "game", "aspect", "sentence_text", "review_score",
                   "p_aspect", "sentiment", "p_negative"]]
    audit["human_label"] = ""
    audit.to_csv(AUDIT_PATH, index=False)
    print(f"Gernerted {len(audit)} pairs -> {AUDIT_PATH}")
    print(audit["aspect"].value_counts().to_string())
    print("Each game:", audit["game"].value_counts().to_dict())
    return audit

audit = generate_audit()

Annotation tools

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def audit_sentiment():
    df = pd.read_csv(AUDIT_PATH)
    df["human_label"] = df["human_label"].fillna("").astype(str)
    state = {"i": None, "hist": []}
    head, body = widgets.HTML(), widgets.HTML()

    def pending():
        return df.index[df["human_label"] == ""].tolist()

    def show():
        t = pending()
        if not t:
            head.value, body.value, state["i"] = "<h3>all done 🎉</h3>", "", None
            return
        i = t[0]
        state["i"] = i
        r = df.loc[i]
        meta = ASPECT_TAXONOMY[r["aspect"]]
        head.value = f"<b>progress {len(df) - len(t)}/{len(df)}</b>"
        body.value = (f"<p style='font-size:19px'>{r['sentence_text']}</p>"
                      f"<p style='color:#8fd0ff'>Aspect：<b>{meta['label']}</b><br>"
                      f"<span style='font-size:12px;color:#aaa'>{meta['description']}</span></p>")

    def save():
        df.to_csv(AUDIT_PATH, index=False)

    def set_label(v):
        if state["i"] is None:
            return
        df.at[state["i"], "human_label"] = v
        state["hist"].append(state["i"])
        save()
        show()

    def undo(_):
        if not state["hist"]:
            return
        i = state["hist"].pop()
        df.at[i, "human_label"] = ""
        save()
        show()

    buttons = []
    for v, style, text in [("negative", "danger", "negative"),
                           ("neutral", "", "neutral"),
                           ("positive", "success", "positive"),
                           ("na", "warning", "We aren't really discussing this aspect.")]:
        b = widgets.Button(description=text, button_style=style, layout=widgets.Layout(width="200px"))
        b.on_click(lambda _, v=v: set_label(v))
        buttons.append(b)
    u = widgets.Button(description="↩ Undo the last action", layout=widgets.Layout(width="140px"))
    u.on_click(undo)
    buttons.append(u)

    show()
    display(widgets.VBox([head, body, widgets.HBox(buttons)]))

audit_sentiment()

Evaluation

In [ ]:
def evaluate_audit():
    aud = pd.read_csv(AUDIT_PATH)
    aud["human_label"] = aud["human_label"].fillna("").astype(str)
    aud = aud[aud["human_label"].isin(["negative", "neutral", "positive", "na"])].copy()
    if len(aud) == 0:
        print("It hasn't been marked in the audit documents yet; run it for now. audit_sentiment()")
        return None
    print(f"Annoated: {len(aud)}   including na (Task 2 false positive) : {int((aud['human_label'] == 'na').sum())}")

    # ---------- A. Task 2 Precision ----------
    aud["valid"] = (aud["human_label"] != "na")
    print("\n=== A1. Task 2 Precision（Various aspects; stratified sampling based on star ratings—for reference only.）===")
    print(aud.groupby("aspect")["valid"].agg(n="size", precision="mean").round(2).to_string())

    aud["p_band"] = pd.cut(aud["p_aspect"], [0, 0.5, 0.75, 1.0001], labels=["<0.5", "0.5-0.75", ">=0.75"])
    print("\n=== A2. Precision by Task 2 probability tier (n and precision) ===")
    print(aud.groupby(["aspect", "p_band"], observed=True)["valid"]
             .agg(["size", "mean"]).round(2).unstack("p_band").to_string())

    # ---------- B. Comparison of the current Task 3 with manual results (binary classification: negative vs. non-negative) ----------
    lab = aud[aud["human_label"].isin(["negative", "neutral", "positive"])].copy()
    lab["y_neg"] = (lab["human_label"] == "negative").astype(int)
    lab["pred_neg"] = (lab["sentiment"] == "negative").astype(int)

    print("\n=== B. Current Task 3 (negative classification threshold: 0.5) ===")
    rows = []
    for a, d in lab.groupby("aspect"):
        rows.append({"aspect": a, "n": len(d),
                     "Proportion of manually identified negatives": round(d["y_neg"].mean(), 2),
                     "Proportion of negative samples in the model": round(d["pred_neg"].mean(), 2),
                     "negative recall": round(((d.pred_neg == 1) & (d.y_neg == 1)).sum() / max(d.y_neg.sum(), 1), 2),
                     "accuracy": round((d.pred_neg == d.y_neg).mean(), 2)})
    print(pd.DataFrame(rows).to_string(index=False))

    # ---------- C. Threshold scanning ----------
    thr = [0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]
    rows = []
    for a, d in lab.groupby("aspect"):
        if len(d) < 20:
            continue
        accs = [((d["p_negative"] >= t).astype(int) == d["y_neg"]).mean() for t in thr]
        rows.append(pd.Series(accs, index=thr, name=a))
    if rows:
        C = pd.DataFrame(rows)
        print("\n=== C1. Threshold scanning：If p_negative >= t, classify as negative (value is accuracy）===")
        print(C.round(2).to_string())
        sm = C.T.rolling(3, center=True, min_periods=1).mean().T        # Smooth to avoid selecting a single-point optimum
        best_t = sm.idxmax(axis=1)
        print("\n=== C2. Recommended threshold (take the highest value after smoothing, choose a plateau range rather than a single optimal point）===")
        print(pd.DataFrame({"n": lab.groupby("aspect").size().reindex(C.index),
                            "acc@0.5": C[0.5].round(2),
                            "Recommended threshold": best_t,
                            "The threshold accuracy": [round(C.loc[a, best_t[a]], 2) for a in C.index]}).to_string())
    else:
        print("\n（The number of annotations for each aspect is fewer than 20, so threshold scanning will not be performed for now.）")
    return aud

_ = evaluate_audit()

Incorporate audit annotations into the training process and conduct rigorous evaluations.

In [ ]:
import math, gc, time
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, DataCollatorWithPadding, set_seed)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

assert torch.cuda.is_available(), "Need GPU"
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)

SENT_LABELS = ["negative", "neutral", "positive"]
L2I = {l: i for i, l in enumerate(SENT_LABELS)}

# ---- gold pairs ----
g = pd.read_csv(f'{PROJECT_ROOT}/data/annotation_v2/gold_set_combined.csv')
rows = []
for _, r in g.iterrows():
    for a in ASPECT_IDS:
        if pd.to_numeric(r.get(f"gold_{a}"), errors="coerce") == 1:
            s = r.get(f"gold_{a}_sentiment")
            if isinstance(s, str) and s in L2I:
                rows.append({"sentence_uid": r["sentence_uid"], "game": r["game"], "aspect": a,
                             "sentence_text": r["sentence_text"], "label": s, "source": "gold"})
gold_pairs = pd.DataFrame(rows)

# ---- The audit pairs (will retain the predictions from the old model for comparison before and after the fix）----
aud = pd.read_csv(AUDIT_PATH)
aud["human_label"] = aud["human_label"].fillna("").astype(str)
aud = aud[aud["human_label"].isin(SENT_LABELS)].copy()
audit_pairs = aud.rename(columns={"human_label": "label"})[
    ["sentence_uid", "game", "aspect", "sentence_text", "label", "sentiment", "p_negative"]]
audit_pairs["source"] = "audit"

data = pd.concat([gold_pairs, audit_pairs], ignore_index=True)
data["y"] = data["label"].map(L2I)
print(data.groupby(["source", "label"]).size().unstack(fill_value=0))
print("Sentences:", data["sentence_uid"].nunique(), " Pairs:", len(data))

Define and run the cross-validation function

In [ ]:
class PairDS(Dataset):
    def __init__(self, df):
        self.enc = tokenizer(list(df["sentence_text"]),
                             [ASPECT_TAXONOMY[a]["label"] for a in df["aspect"]],
                             truncation=True, max_length=96)
        self.y = df["y"].tolist()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.y[i]
        return item

class WTrainer(Trainer):
    def __init__(self, cw=None, **kw):
        super().__init__(**kw)
        self.cw = cw
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=self.cw.to(out.logits.device))(out.logits, labels)
        return (loss, out) if return_outputs else loss

def soft_weights(y):
    cnt = np.bincount(y, minlength=3).astype(float)
    cw = len(y) / (3 * np.maximum(cnt, 1))              # Balancing weights
    return torch.tensor(np.clip(np.sqrt(cw), 0.5, 3.0), dtype=torch.float)   # Apply square root and limit the amplitude to avoid overcompensation.

def train_predict(train_df, test_df, seed, epochs=6, bs=16, lr=2e-5):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    steps = math.ceil(len(train_df) / bs) * epochs
    args = TrainingArguments(output_dir="/content/t3cv_tmp", num_train_epochs=epochs,
                             per_device_train_batch_size=bs, per_device_eval_batch_size=64,
                             learning_rate=lr, warmup_steps=int(0.1 * steps), weight_decay=0.01,
                             eval_strategy="no", save_strategy="no", logging_strategy="no",
                             fp16=True, report_to="none", disable_tqdm=True, seed=seed)
    tr = WTrainer(cw=soft_weights(train_df["y"].values), model=model, args=args,
                  train_dataset=PairDS(train_df), data_collator=collator)
    tr.train()
    logits = tr.predict(PairDS(test_df)).predictions
    del tr, model
    gc.collect(); torch.cuda.empty_cache()
    e = np.exp(logits - logits.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

def run_cv(mode="group", n_splits=5, seed=42):
    oof = np.zeros((len(data), 3))
    if mode == "group":       # Grouped by sentence; multiple aspects belonging to the same sentence do not span across the training and test sets.
        sp = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        splits = list(sp.split(data, data["y"], groups=data["sentence_uid"]))
    else:                     # Keep one game to directly test whether there is a reliance on game shortcuts.
        splits = [(np.where(data["game"] != g_)[0], np.where(data["game"] == g_)[0])
                  for g_ in GAME_SLUGS]
    for k, (tr_i, te_i) in enumerate(splits, 1):
        t0 = time.time()
        oof[te_i] = train_predict(data.iloc[tr_i], data.iloc[te_i], seed + k)
        print(f"[{mode}] fold {k}/{len(splits)}  train={len(tr_i)}  test={len(te_i)}  "
              f"{(time.time() - t0) / 60:.1f} mins")
    np.save(f'{PROJECT_ROOT}/outputs/task3_oof_{mode}.npy', oof)
    return oof

oof_group = run_cv("group")

Evaluation (comparison before and after repair)

In [ ]:
def evaluate_oof(oof, title):
    d = data.copy()
    d["new_pneg"] = oof[:, 0]
    d["new_neg"] = (oof.argmax(1) == 0).astype(int)
    d["y_neg"] = (d["y"] == 0).astype(int)

    def auc(y, p):
        return round(roc_auc_score(y, p), 2) if 0 < y.sum() < len(y) else np.nan

    a = d[d["source"] == "audit"].copy()
    a["old_neg"] = (a["sentiment"] == "negative").astype(int)

    def summarize(x):
        return {"n": len(x), "human_neg": round(x.y_neg.mean(), 2),
                "baseline": round(max(x.y_neg.mean(), 1 - x.y_neg.mean()), 2),
                "old_acc": round((x.old_neg == x.y_neg).mean(), 2),
                "new_acc": round((x.new_neg == x.y_neg).mean(), 2),
                "old_AUC": auc(x.y_neg, x.p_negative), "new_AUC": auc(x.y_neg, x.new_pneg),
                "old_neg%": round(x.old_neg.mean(), 2), "new_neg%": round(x.new_neg.mean(), 2)}

    print(f"===== {title}：Audit pairs，Based on aspect (negative vs. non-negative) =====")
    t = pd.DataFrame({asp: summarize(x) for asp, x in a.groupby("aspect")}).T
    print(t.to_string())
    print("\nAll audit pairs:", summarize(a))

    print("\nGroup by game:")
    print(pd.DataFrame({gm: summarize(x) for gm, x in a.groupby("game")}).T
            [["n", "human_neg", "old_acc", "new_acc", "old_neg%", "new_neg%"]].to_string())

    gg = d[d["source"] == "gold"]
    print(f"\ngold negative-vs-rest accuracy(Out-of-sample) : {(gg.new_neg == gg.y_neg).mean():.3f}   "
          f"AUC: {auc(gg.y_neg, gg.new_pneg)}")

evaluate_oof(oof_group, "5 folds（Group by sentence）")

Training for the final Task 3 (gold + audit, totaling 604 pairs)

In [ ]:
set_seed(42)
FINAL3_DIR = f'{PROJECT_ROOT}/models/task3_final_v2'
os.makedirs(FINAL3_DIR, exist_ok=True)

model_f = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3,
    id2label={i: l for i, l in enumerate(SENT_LABELS)},
    label2id=L2I)
EPOCHS, BS = 6, 16
args_f = TrainingArguments(
    output_dir="/content/t3_final_tmp", num_train_epochs=EPOCHS,
    per_device_train_batch_size=BS, learning_rate=2e-5,
    warmup_steps=int(0.1 * math.ceil(len(data) / BS) * EPOCHS), weight_decay=0.01,
    eval_strategy="no", save_strategy="no", logging_steps=20,
    fp16=True, report_to="none", seed=42)
tr_f = WTrainer(cw=soft_weights(data["y"].values), model=model_f, args=args_f,
                train_dataset=PairDS(data), data_collator=collator)
tr_f.train()
tr_f.save_model(FINAL3_DIR)
tokenizer.save_pretrained(FINAL3_DIR)
print("Saved ->", FINAL3_DIR)

Re-annotate the sentiment of all sentence-aspect pairs using the new model

In [ ]:
CFG["task3_model_dir"] = "models/task3_final_v2"
TASK3_DIR = f'{PROJECT_ROOT}/{CFG["task3_model_dir"]}'

In [ ]:
import shutil, time
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ---------- 0. Configuration: Switch Task 3 to the new model; revert the threshold for Task 2 to 0.30. ----------
CFG["task3_model_dir"] = "models/task3_final_v2"
CFG.pop("task2_threshold_per_aspect", None)          # Revert the previous individual setting of content=0.75 (audit indicates it is unnecessary)
with open(cfg_path, "w") as f:
    json.dump(CFG, f, indent=2)

TASK3_DIR = f'{PROJECT_ROOT}/{CFG["task3_model_dir"]}'
THRESHOLD = CFG["task2_threshold"]
assert os.path.exists(f'{TASK3_DIR}/model.safetensors'), f"New model not found：{TASK3_DIR}，confirm that task3_final_v2 has been trained and saved."
print("Task3 model:", TASK3_DIR, "| Task2 threshold:", THRESHOLD)

# ---------- 1. Back up the old pairs (to serve as a comparison before and after the fix; back up only once) ----------
BK = f'{PROJECT_ROOT}/outputs/predictions_v1'
os.makedirs(BK, exist_ok=True)
for slug in GAME_SLUGS:
    src = f'{PROJECT_ROOT}/outputs/predictions/{slug}_pairs.parquet'
    dst = f'{BK}/{slug}_pairs.parquet'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
print("The old version of Pairs is backed up at:", BK)

# ---------- 2. Load the predictions from Task 2 and re-evaluate them using a unified threshold ----------
preds_all = {}
for slug in GAME_SLUGS:
    p = f'{PROJECT_ROOT}/outputs/predictions/{slug}_aspects.parquet'
    out = pd.read_parquet(p)
    app = applicable_vector(slug)
    for j, a in enumerate(ASPECT_IDS):
        out[f"pred_{a}"] = (out[f"p_{a}"].values >= THRESHOLD) & app[j]
    out.to_parquet(p, index=False)
    preds_all[slug] = out
    print(f"{slug:12s} {len(out):6d} sentences | logarithm of the pair judged to have an aspect: "
          f"{int(sum(out[f'pred_{a}'].sum() for a in ASPECT_IDS))}")

# ---------- 3. Loading Task 3 ----------
tok3 = AutoTokenizer.from_pretrained(TASK3_DIR)
m3 = AutoModelForSequenceClassification.from_pretrained(TASK3_DIR).to(device).eval()
SENT_LABELS = [m3.config.id2label[i] for i in range(m3.config.num_labels)]
assert SENT_LABELS == ["negative", "neutral", "positive"], SENT_LABELS
print("Task3 loaded ✅", SENT_LABELS)

@torch.inference_mode()
def predict_logits(texts, tok, model, batch_size=128, max_length=96, pair_texts=None):
    order = np.argsort([len(t) for t in texts])
    out = None
    for start in range(0, len(texts), batch_size):
        idx = order[start:start + batch_size]
        batch = [texts[i] for i in idx]
        if pair_texts is None:
            enc = tok(batch, truncation=True, max_length=max_length, padding=True, return_tensors="pt")
        else:
            enc = tok(batch, [pair_texts[i] for i in idx], truncation=True,
                      max_length=max_length, padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**enc).logits
        logits = logits.float().cpu().numpy()
        if out is None:
            out = np.zeros((len(texts), logits.shape[1]), dtype=np.float32)
        out[idx] = logits
    return out

# ---------- 4. reasoning ----------
def run_task3(slug):
    t0 = time.time()
    out = preds_all[slug]
    parts = []
    for a in ASPECT_IDS:
        idx = np.where(out[f"pred_{a}"].values)[0]
        if len(idx):
            parts.append(pd.DataFrame({"row": idx, "aspect": a}))
    pairs = pd.concat(parts, ignore_index=True)

    texts = out["sentence_text"].values[pairs["row"].values].tolist()
    aspect_text = [ASPECT_TAXONOMY[a]["label"] for a in pairs["aspect"]]   # It must match the second text segment used during training
    logits = predict_logits(texts, tok3, m3, pair_texts=aspect_text)
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = e / e.sum(axis=1, keepdims=True)

    meta_cols = [c for c in ["sentence_uid", "reviewId", "game", "sentence_text", "review_score",
                             "reviewCreatedVersion", "at"] if c in out.columns]
    res = out[meta_cols].iloc[pairs["row"].values].reset_index(drop=True)
    res["aspect"] = pairs["aspect"].values
    res["p_aspect"] = [out[f"p_{a}"].values[r] for a, r in zip(pairs["aspect"], pairs["row"])]
    res["sentiment"] = [SENT_LABELS[i] for i in probs.argmax(axis=1)]
    for j, lab in enumerate(SENT_LABELS):
        res[f"p_{lab}"] = probs[:, j].astype(np.float32)

    res.to_parquet(f'{PROJECT_ROOT}/outputs/predictions/{slug}_pairs.parquet', index=False)
    print(f"{slug:12s} {len(res):6d} (sentences, aspect)pairs | {(time.time() - t0) / 60:.1f} mins")
    return res

pairs_all = {slug: run_task3(slug) for slug in GAME_SLUGS}

# ---------- 5. Before-and-after comparison of the repair ----------
old = pd.concat([pd.read_parquet(f'{BK}/{s}_pairs.parquet') for s in GAME_SLUGS], ignore_index=True)
new = pd.concat(pairs_all.values(), ignore_index=True)
print("\n pairs old/new:", len(old), len(new), "（They should be the same, because the result of Task 2 hasn't changed）")

neg = lambda s: (s == "negative").mean()
cmp = pd.DataFrame({
    "n": new.groupby(["game", "aspect"]).size(),
    "Old model negative proportion": old.groupby(["game", "aspect"])["sentiment"].apply(neg),
    "New model negative proportion": new.groupby(["game", "aspect"])["sentiment"].apply(neg),
}).round(3)

focus = ["gacha_rates", "content_progression", "monetization_value",
         "character_operator_balance", "level_difficulty_balance"]
print("\n=== Key aspect: Percentage of negative results between the old and new models (by game）===")
print(cmp[cmp.index.get_level_values("aspect").isin(focus)].sort_index(level=["aspect", "game"]).to_string())

print("\n=== Overall negative percentage for each aspect (all games combined）===")
print(pd.DataFrame({
    "Old": old.groupby("aspect")["sentiment"].apply(neg),
    "New": new.groupby("aspect")["sentiment"].apply(neg),
    "n": new.groupby("aspect").size()}).round(3).sort_values("n", ascending=False).to_string())

Compare the negative rates of human annotators and models by star rating

In [ ]:
aud = pd.read_csv(AUDIT_PATH)
aud["human_label"] = aud["human_label"].fillna("").astype(str)
aud = aud[aud["human_label"].isin(["negative", "neutral", "positive"])].copy()
band = lambda s: np.where(s <= 2, "1-2", np.where(s >= 4, "4-5", "3"))
aud["band"] = band(pd.to_numeric(aud["review_score"], errors="coerce"))
aud["h_neg"] = (aud["human_label"] == "negative")

new = pd.concat(pairs_all.values(), ignore_index=True)
new["band"] = band(pd.to_numeric(new["review_score"], errors="coerce"))
new["m_neg"] = (new["sentiment"] == "negative")

focus = ["level_difficulty_balance", "monetization_value", "content_progression",
         "gacha_rates", "character_operator_balance"]
rows = []
for a in focus:
    for b in ["1-2", "4-5"]:
        h = aud[(aud.aspect == a) & (aud.band == b)]
        m = new[(new.aspect == a) & (new.band == b)]
        rows.append({"aspect": a, "stars": b, "manual n": len(h),
                     "Manual false-negative rate": round(h.h_neg.mean(), 2) if len(h) else np.nan,
                     "Full model n": len(m), "Negative rate of the full model": round(m.m_neg.mean(), 2)})
print(pd.DataFrame(rows).to_string(index=False))

Read 15 sentences—specifically those from 4- or 5-star reviews that were classified as "negative" regarding "level_difficulty"—and determine how many of them are truly negative

In [ ]:
d = new[(new.aspect == "level_difficulty_balance") & (new.band == "4-5") & new.m_neg]
for _, r in d.sample(min(15, len(d)), random_state=3).iterrows():
    print(f"p_neg={r['p_negative']:.2f} | {r['game']:10s} | {r['sentence_text'][:130]}")

#### issue profile

In [ ]:
if "pairs_all" not in globals():
    pairs_all = {s: pd.read_parquet(f'{PROJECT_ROOT}/outputs/predictions/{s}_pairs.parquet') for s in GAME_SLUGS}
if "preds_all" not in globals():
    preds_all = {s: pd.read_parquet(f'{PROJECT_ROOT}/outputs/predictions/{s}_aspects.parquet',
                                    columns=["reviewId", "sentence_uid"]) for s in GAME_SLUGS}

MIN_MENTIONS = 30
EXCLUDE = {"grind_progression"}          # There are only 4 training samples; Task 2 cannot be evaluated and is excluded from the ranking.
CAUTION = {"content_progression": "Widest boundaries, lowest Task 2 AP.",
           "character_operator_balance": "Annotation definition drift; performance is weak on both Task 2 and Task 3.",
           "level_difficulty_balance": "Sentiment judgment pending verification"}

def review_matrices(slug):
    sents = preds_all[slug]
    pr = pairs_all[slug][["reviewId", "aspect", "sentiment", "p_negative"]].copy()
    rid = pd.Index(sents["reviewId"].unique())
    S = sents.groupby("reviewId").size().reindex(rid).values.astype(float)
    pr["one"] = 1.0
    pr["hard"] = (pr["sentiment"] == "negative").astype(float)
    def mat(col):
        m = pr.pivot_table(index="reviewId", columns="aspect", values=col, aggfunc="sum")
        return m.reindex(index=rid, columns=ASPECT_IDS).fillna(0.0).values
    return S, mat("one"), mat("hard"), mat("p_negative")

def issue_profile(slug, B=300, seed=0):
    S, M, H, P = review_matrices(slug)
    R, tot = len(S), S.sum()
    rng = np.random.default_rng(seed)
    men, hard, soft = M.sum(0), H.sum(0), P.sum(0)
    df = pd.DataFrame({"game": slug, "aspect": ASPECT_IDS, "mentions": men.astype(int),
                       "neg_rate": hard / np.maximum(men, 1),
                       "prio": 100 * hard / tot, "prio_soft": 100 * soft / tot})
    boot = np.array([100 * H[ix].sum(0) / S[ix].sum()
                     for ix in (rng.integers(0, R, R) for _ in range(B))])
    df["ci_lo"], df["ci_hi"] = np.percentile(boot, [2.5, 97.5], axis=0)
    ok = (df["mentions"] >= MIN_MENTIONS) & ~df["aspect"].isin(EXCLUDE) & applicable_vector(slug)
    idx_ok = np.where(ok)[0]
    top3 = np.zeros(len(ASPECT_IDS))
    for b in range(B):
        top3[idx_ok[np.argsort(-boot[b, idx_ok])[:3]]] += 1
    df["P_top3"] = top3 / B
    df["rank"] = df.loc[ok, "prio"].rank(ascending=False)
    df["rank_soft"] = df.loc[ok, "prio_soft"].rank(ascending=False)
    df["note"] = df["aspect"].map(CAUTION).fillna("")
    return df[ok].sort_values("rank")

profiles = pd.concat([issue_profile(s) for s in GAME_SLUGS], ignore_index=True)
profiles.to_csv(f'{PROJECT_ROOT}/outputs/issue_profile_all.csv', index=False)

show = ["aspect", "mentions", "neg_rate", "prio", "ci_lo", "ci_hi", "rank", "rank_soft", "P_top3", "note"]
for s in GAME_SLUGS:
    print(f"\n===== {s} =====")
    print(profiles[profiles.game == s][show].round(2).to_string(index=False))

cg = profiles[profiles.aspect.isin(COMMON)]
print("\n=== Cross-game comparison (general aspects only): Number of negative mentions per 100 sentences ===")
print(cg.pivot(index="aspect", columns="game", values="prio").round(2).to_string())
print("\n=== Cross-game comparison: proportion of negative sentiment ===")
print(cg.pivot(index="aspect", columns="game", values="neg_rate").round(2).to_string())

Calibrate level_difficulty: No retraining required

In [ ]:
# ========== 1. Rebuild the data in the training notebook (the order must be consistent with the original）==========
if "AUDIT_PATH" not in globals():
    AUDIT_PATH = f'{PROJECT_ROOT}/outputs/sentiment_audit.csv'

LABELS3 = ["negative", "neutral", "positive"]
L2I = {l: i for i, l in enumerate(LABELS3)}

g = pd.read_csv(f'{PROJECT_ROOT}/data/annotation_v2/gold_set_combined.csv')
rows = []
for _, r in g.iterrows():
    for a in ASPECT_IDS:
        if pd.to_numeric(r.get(f"gold_{a}"), errors="coerce") == 1:
            s = r.get(f"gold_{a}_sentiment")
            if isinstance(s, str) and s in L2I:
                rows.append({"sentence_uid": r["sentence_uid"], "game": r["game"], "aspect": a,
                             "sentence_text": r["sentence_text"], "label": s, "source": "gold"})
gold_pairs = pd.DataFrame(rows)

aud_ = pd.read_csv(AUDIT_PATH)
aud_["human_label"] = aud_["human_label"].fillna("").astype(str)
aud_ = aud_[aud_["human_label"].isin(LABELS3)].copy()
audit_pairs = aud_.rename(columns={"human_label": "label"})[
    ["sentence_uid", "game", "aspect", "sentence_text", "label", "sentiment", "p_negative"]]
audit_pairs["source"] = "audit"

cal_data = pd.concat([gold_pairs, audit_pairs], ignore_index=True)
cal_data["y"] = cal_data["label"].map(L2I)

# ========== 2. Load the out-of-sample predictions and verify that the row order matches ==========
oof_group = np.load(f'{PROJECT_ROOT}/outputs/task3_oof_group.npy')
assert len(oof_group) == len(cal_data), f"Inconsistent lengths：oof={len(oof_group)} data={len(cal_data)}"

cal_data["pneg"] = oof_group[:, 0]
cal_data["y_neg"] = (cal_data["y"] == 0).astype(int)

# Verification: The out-of-sample accuracy against the audit set should be approximately equal to the previously calculated 0.79; against the gold set, it is approximately 0.83.
chk_a = cal_data[cal_data.source == "audit"]
chk_g = cal_data[cal_data.source == "gold"]
acc_a = ((oof_group[chk_a.index, 0] == oof_group[chk_a.index].max(1)).astype(int) == chk_a.y_neg).mean()
acc_g = ((oof_group[chk_g.index, 0] == oof_group[chk_g.index].max(1)).astype(int) == chk_g.y_neg).mean()
print(f"Validation: Auditing out-of-sample accuracy {acc_a:.3f}（around0.79），gold pairs {acc_g:.3f}（around0.83）")
print("Total rows:", len(cal_data), "（around604）")

# ========== 3. Threshold scan for level_difficulty ==========
lv = cal_data[cal_data["aspect"] == "level_difficulty_balance"]
print("\nlevel_difficulty samples:", len(lv),
      "| Proportion of manually identified negatives:", round(lv.y_neg.mean(), 2),
      "（gold", int((lv.source == 'gold').sum()), "audit", int((lv.source == 'audit').sum()), "）")

rows = []
for t in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]:
    pr = (lv["pneg"] >= t).astype(int)
    tp = int(((pr == 1) & (lv.y_neg == 1)).sum())
    rows.append({"threshold": t,
                 "accuracy": round((pr == lv.y_neg).mean(), 2),
                 "negative_recall": round(tp / max(lv.y_neg.sum(), 1), 2),
                 "negative_precision": round(tp / max(pr.sum(), 1), 2),
                 "pred_neg_rate": round(pr.mean(), 2)})
print(pd.DataFrame(rows).to_string(index=False))

Report on negative mentions of `level_difficulty`, broken down by star rating

In [ ]:
if "pairs_all" not in globals():
    pairs_all = {s: pd.read_parquet(f'{PROJECT_ROOT}/outputs/predictions/{s}_pairs.parquet') for s in GAME_SLUGS}
new = pd.concat(pairs_all.values(), ignore_index=True)
new["stars"] = pd.to_numeric(new["review_score"], errors="coerce")
new["m_neg"] = (new["sentiment"] == "negative")

tot = pd.concat([pd.read_parquet(f'{PROJECT_ROOT}/outputs/predictions/{s}_aspects.parquet',
                                 columns=["game", "review_score"]) for s in GAME_SLUGS])
tot["stars"] = pd.to_numeric(tot["review_score"], errors="coerce")

def per100(mask_pairs, mask_tot, game):
    n_pairs = int((mask_pairs & (new.game == game)).sum())
    n_sent = int((mask_tot & (tot.game == game)).sum())
    return round(100 * n_pairs / n_sent, 2)

lv_ = new.aspect == "level_difficulty_balance"
rows = []
for gname in GAME_SLUGS:
    rows.append({"game": gname,
                 "All comments negative mention/100 sentences": per100(lv_ & new.m_neg, tot.stars.notna() | tot.stars.isna(), gname),
                 "only 1-2 stars comments negative mention/100 sentences": per100(lv_ & new.m_neg & (new.stars <= 2), tot.stars <= 2, gname),
                 "only 4-5 stars comments negative mention/100 sentences": per100(lv_ & new.m_neg & (new.stars >= 4), tot.stars >= 4, gname)})
print(pd.DataFrame(rows).to_string(index=False))

Robustness view, no retraining required

In [ ]:
if "pairs_all" not in globals():
    pairs_all = {s: pd.read_parquet(f'{PROJECT_ROOT}/outputs/predictions/{s}_pairs.parquet') for s in GAME_SLUGS}
new = pd.concat(pairs_all.values(), ignore_index=True)
new["stars"] = pd.to_numeric(new["review_score"], errors="coerce")
new["m_neg"] = new["sentiment"].eq("negative")

tot = pd.concat([pd.read_parquet(f'{PROJECT_ROOT}/outputs/predictions/{s}_aspects.parquet',
                                 columns=["game", "review_score"]) for s in GAME_SLUGS])
tot["stars"] = pd.to_numeric(tot["review_score"], errors="coerce")

rows = []
for gname in GAME_SLUGS:
    n_all = int((tot.game == gname).sum())
    n_low = int(((tot.game == gname) & (tot.stars <= 2)).sum())
    d = new[(new.game == gname) & new.m_neg]
    df = pd.DataFrame({
        "prio_all": d.groupby("aspect").size() * 100 / n_all,
        "prio_1to2": d[d.stars <= 2].groupby("aspect").size() * 100 / n_low,
        "mentions": new[new.game == gname].groupby("aspect").size(),
    }).fillna(0)
    app = applicable_vector(gname)
    keep = [a for a in ASPECT_IDS if app[ASPECT_IDS.index(a)] and a != "grind_progression"]
    df = df.reindex(keep).fillna(0)
    df = df[df["mentions"] >= 30]
    df["rank_all"] = df["prio_all"].rank(ascending=False)
    df["rank_1to2"] = df["prio_1to2"].rank(ascending=False)
    df["game"] = gname
    rows.append(df.reset_index().rename(columns={"index": "aspect"}))
rob = pd.concat(rows, ignore_index=True)
rob.to_csv(f'{PROJECT_ROOT}/outputs/issue_profile_robustness.csv', index=False)

for gname in GAME_SLUGS:
    d = rob[rob.game == gname].sort_values("rank_all")
    rho = d[["rank_all", "rank_1to2"]].corr(method="spearman").iloc[0, 1]
    top_all = set(d.nsmallest(3, "rank_all")["aspect"])
    top_low = set(d.nsmallest(3, "rank_1to2")["aspect"])
    print(f"\n===== {gname} =====   Ranking-related (Spearman)={rho:.2f}   The top three overlap: {len(top_all & top_low)}/3")
    print(d[["aspect", "prio_all", "rank_all", "prio_1to2", "rank_1to2"]].round(2).to_string(index=False))

Export

In [ ]:
import os, json
import numpy as np, pandas as pd

OUT = f'{PROJECT_ROOT}/outputs/dashboard'
os.makedirs(OUT, exist_ok=True)

# ---------- Read ----------
pairs = pd.concat([pd.read_parquet(f'{PROJECT_ROOT}/outputs/predictions/{s}_pairs.parquet') for s in GAME_SLUGS],
                  ignore_index=True)
sents = pd.concat([pd.read_parquet(f'{PROJECT_ROOT}/outputs/predictions/{s}_aspects.parquet',
                                   columns=["sentence_uid", "reviewId", "game", "review_score", "at"])
                   for s in GAME_SLUGS], ignore_index=True)
for d in (pairs, sents):
    d["stars"] = pd.to_numeric(d["review_score"], errors="coerce")
    d["at"] = pd.to_datetime(d["at"], errors="coerce")
pairs["m_neg"] = pairs["sentiment"].eq("negative")

# ---------- 1. profile.csv：All reviews + 1–2 stars only ----------
main = pd.read_csv(f'{PROJECT_ROOT}/outputs/issue_profile_all.csv')       # Includes confidence intervals
rob = pd.read_csv(f'{PROJECT_ROOT}/outputs/issue_profile_robustness.csv')[
    ["game", "aspect", "prio_1to2", "rank_1to2"]]
profile = main.merge(rob, on=["game", "aspect"], how="left")
profile["scope"] = np.where(profile["aspect"].isin(COMMON), "common", "genre-specific")
profile.to_csv(f'{OUT}/profile.csv', index=False)
print("profile.csv:", profile.shape)

# ---------- 2. examples.parquet：Representative negative reviews ----------
raw = {}
for s in GAME_SLUGS:
    p = f'{PROJECT_ROOT}/data/processed/{s}_clean.csv'
    if os.path.exists(p):
        r = pd.read_csv(p, usecols=lambda c: c in ["reviewId", "content_clean", "thumbsUpCount"])
        raw[s] = r
rev = pd.concat(raw.values(), ignore_index=True) if raw else pd.DataFrame(columns=["reviewId"])
rev = rev.drop_duplicates("reviewId")

ex = pairs[pairs["m_neg"]].merge(rev, on="reviewId", how="left")
ex["score"] = ex["p_aspect"] * ex["p_negative"]
if "thumbsUpCount" in ex.columns:
    ex["thumbs"] = pd.to_numeric(ex["thumbsUpCount"], errors="coerce").fillna(0)
else:
    ex["thumbs"] = 0
ex = (ex.sort_values(["game", "aspect", "score", "thumbs"], ascending=[True, True, False, False])
        .groupby(["game", "aspect"]).head(30))
keep = [c for c in ["game", "aspect", "sentence_text", "content_clean", "review_score",
                    "p_aspect", "p_negative", "score", "thumbs", "reviewCreatedVersion", "at"]
        if c in ex.columns]
exo = ex[keep].copy()

if "reviewCreatedVersion" in exo.columns:
    exo["reviewCreatedVersion"] = exo["reviewCreatedVersion"].astype("string").fillna("")

for c in ["game", "aspect", "sentence_text", "content_clean"]:
    if c in exo.columns:
        exo[c] = exo[c].astype("string").fillna("")

exo["at"] = pd.to_datetime(exo["at"], errors="coerce")
exo["review_score"] = pd.to_numeric(exo["review_score"], errors="coerce")

exo.to_parquet(f'{OUT}/examples.parquet', index=False)
print("examples.parquet:", exo.shape)

# ---------- 3. trend.csv：Rate of negative mentions per two-week window ----------
sents["window"] = sents["at"].dt.to_period("2W").dt.start_time
pairs["window"] = pd.to_datetime(pairs["at"], errors="coerce").dt.to_period("2W").dt.start_time
den = sents.groupby(["game", "window"]).size().rename("n_sentences")
num = pairs[pairs["m_neg"]].groupby(["game", "aspect", "window"]).size().rename("n_neg")
trend = num.reset_index().merge(den.reset_index(), on=["game", "window"])
trend["neg_per100"] = 100 * trend["n_neg"] / trend["n_sentences"]
trend = trend[trend["n_sentences"] >= 100]                         # Window too small and unreliable; discarding.
trend.to_csv(f'{OUT}/trend.csv', index=False)
print("trend.csv:", trend.shape, " time range:", sents["at"].min(), "→", sents["at"].max())

Task 1 Overall Sentiment (Home Page Overview)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

T1 = f'{PROJECT_ROOT}/models/task1_bert_combined'
tok1 = AutoTokenizer.from_pretrained(T1)
m1 = AutoModelForSequenceClassification.from_pretrained(T1).to(device).eval()
assert [m1.config.id2label[i] for i in range(3)] == ["negative", "neutral", "positive"]

ov = []
for s in GAME_SLUGS:
    r = pd.read_csv(f'{PROJECT_ROOT}/data/processed/{s}_clean.csv',
                    usecols=["reviewId", "content_clean", "score", "at"]).dropna(subset=["content_clean"])
    logits = predict_logits(r["content_clean"].astype(str).tolist(), tok1, m1, batch_size=128, max_length=256)
    r["overall"] = np.array(["negative", "neutral", "positive"])[logits.argmax(1)]
    ov.append({"game": s, "n_reviews": len(r), "avg_stars": round(pd.to_numeric(r["score"]).mean(), 2),
               "pct_negative": round((r["overall"] == "negative").mean(), 3),
               "pct_neutral": round((r["overall"] == "neutral").mean(), 3),
               "pct_positive": round((r["overall"] == "positive").mean(), 3)})
    r[["reviewId", "overall"]].to_parquet(f'{OUT}/overall_{s}.parquet', index=False)
overview = pd.DataFrame(ov)
overview.to_csv(f'{OUT}/overview.csv', index=False)
print(overview.to_string(index=False))